# Dynamic Pricing & Demand Forecasting Engine

Architecture
------------
1. DemandForecaster
   - Predicts expected demand (units sold) for a product given price, seasonality,
     day-of-week, promotions, and competitor price — using Gradient Boosted Trees,
     which handle the non-linear, interacting effects of pricing well (e.g. demand
     drops fast above a threshold price, weekends behave differently, etc.)

2. PriceElasticityEstimator
   - Estimates how demand responds to price changes for each product by querying
     the trained DemandForecaster across a grid of candidate prices (holding all
     other features fixed) — this produces an estimated demand curve per product,
     analogous to fitting elasticity from real experiment/observational data

3. PriceOptimizer
   - Given a demand curve (from the elasticity estimator) and unit cost, searches
     over candidate prices to find the one that maximizes projected *revenue* or
     *profit* (configurable objective), subject to simple business constraints
     (min/max price bounds, max price change vs current price)
   - This is a classic constrained optimization problem — solved here via grid
     search since the demand curve is a black-box model output (not a closed-form
     function), which is exactly what real-world dynamic pricing systems do at
     the "candidate price scoring" stage

Why this design?
- Forecasting demand and *setting* price are two distinct problems that get
  conflated in naive approaches. This pipeline forecasts demand as a function of
  price (not just future demand at a fixed price), which is what makes true price
  *optimization* possible rather than just demand prediction.
- Grid-search optimization (rather than closed-form calculus) is used deliberately
  because in production the demand model is a black-box ML model, not a simple
  linear/log-linear curve — so gradient-free search over a bounded price grid is
  the standard practical approach.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.ensemble import GradientBoostingRegressor

In [2]:
# ----------------------------------------------------------------------------
# 1. Synthetic sales history (stand-in for a real point-of-sale / e-commerce log)
# ----------------------------------------------------------------------------
def generate_synthetic_data(n_products=15, n_days=365, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    base_prices = rng.uniform(15, 120, size=n_products)
    base_demand = rng.uniform(40, 200, size=n_products)
    elasticity = rng.uniform(-2.5, -0.8, size=n_products)  # negative: price up -> demand down

    for p in range(n_products):
        for day in range(n_days):
            dow = day % 7
            is_weekend = int(dow >= 5)
            season_factor = 1 + 0.3 * np.sin(2 * np.pi * day / 365)  # yearly seasonality
            promo = int(rng.random() < 0.08)
            competitor_price = base_prices[p] * rng.uniform(0.85, 1.15)

            # Simulate a price that varies day to day (past pricing decisions/tests)
            price = base_prices[p] * rng.uniform(0.8, 1.2)

            # True demand curve: base * season * weekend_lift * promo_lift * price_elasticity_effect
            price_ratio = price / base_prices[p]
            demand_mean = (
                base_demand[p]
                * season_factor
                * (1.15 if is_weekend else 1.0)
                * (1.4 if promo else 1.0)
                * (price_ratio ** elasticity[p])
            )
            units_sold = max(0, rng.normal(demand_mean, demand_mean * 0.15))

            rows.append((p, day, dow, is_weekend, season_factor, promo,
                         competitor_price, price, units_sold, base_prices[p]))

    df = pd.DataFrame(rows, columns=[
        "product_id", "day", "day_of_week", "is_weekend", "season_factor", "promo",
        "competitor_price", "price", "units_sold", "base_price"
    ])
    return df

In [3]:
# ----------------------------------------------------------------------------
# 2. Demand Forecasting model
# ----------------------------------------------------------------------------
class DemandForecaster:
    FEATURES = ["product_id", "day_of_week", "is_weekend", "season_factor",
                "promo", "competitor_price", "price"]

    def __init__(self, seed=42):
        self.model = GradientBoostingRegressor(
            n_estimators=300, max_depth=4, learning_rate=0.05, random_state=seed
        )

    def fit(self, df):
        X = df[self.FEATURES]
        y = df["units_sold"]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        self.model.fit(X_train, y_train)
        self.X_test, self.y_test = X_test, y_test
        return self

    def evaluate(self):
        preds = self.model.predict(self.X_test)
        mae = mean_absolute_error(self.y_test, preds)
        mape = mean_absolute_percentage_error(self.y_test.clip(lower=1), preds.clip(min=1))
        print(f"Demand Forecast — MAE: {mae:.2f} units | MAPE: {mape:.1%}")
        return mae, mape

    def predict_demand(self, product_id, price, day_of_week=2, is_weekend=0,
                        season_factor=1.0, promo=0, competitor_price=None):
        if competitor_price is None:
            competitor_price = price  # assume parity if not specified
        row = pd.DataFrame([{
            "product_id": product_id, "day_of_week": day_of_week, "is_weekend": is_weekend,
            "season_factor": season_factor, "promo": promo,
            "competitor_price": competitor_price, "price": price,
        }])
        return max(0, self.model.predict(row[self.FEATURES])[0])

In [4]:
# ----------------------------------------------------------------------------
# 3. Price Elasticity Estimator — probes the demand model across a price grid
# ----------------------------------------------------------------------------
class PriceElasticityEstimator:
    def __init__(self, forecaster):
        self.forecaster = forecaster

    def estimate_curve(self, product_id, price_grid, **context):
        demands = [self.forecaster.predict_demand(product_id, p, **context) for p in price_grid]
        return pd.DataFrame({"price": price_grid, "predicted_demand": demands})

In [5]:
# ----------------------------------------------------------------------------
# 4. Price Optimizer — grid search over candidate prices to maximize revenue/profit
# ----------------------------------------------------------------------------
class PriceOptimizer:
    def __init__(self, forecaster, unit_cost_ratio=0.55):
        """unit_cost_ratio: assumed cost as a fraction of current price (used for profit objective)."""
        self.forecaster = forecaster
        self.unit_cost_ratio = unit_cost_ratio

    def optimize(self, product_id, current_price, objective="profit",
                 max_price_change_pct=0.25, n_candidates=41, **context):
        low = current_price * (1 - max_price_change_pct)
        high = current_price * (1 + max_price_change_pct)
        price_grid = np.linspace(low, high, n_candidates)

        unit_cost = current_price * self.unit_cost_ratio

        best_price, best_value, best_demand = None, -np.inf, None
        results = []
        for price in price_grid:
            demand = self.forecaster.predict_demand(product_id, price, **context)
            revenue = price * demand
            profit = (price - unit_cost) * demand
            value = profit if objective == "profit" else revenue
            results.append((price, demand, revenue, profit))
            if value > best_value:
                best_price, best_value, best_demand = price, value, demand

        results_df = pd.DataFrame(results, columns=["price", "predicted_demand", "revenue", "profit"])
        return {
            "recommended_price": round(best_price, 2),
            "current_price": current_price,
            "price_change_pct": round((best_price - current_price) / current_price * 100, 1),
            "predicted_demand_at_recommended_price": round(best_demand, 1),
            "objective": objective,
            "objective_value": round(best_value, 2),
            "full_grid": results_df,
        }


if __name__ == "__main__":
    df = generate_synthetic_data(n_products=15, n_days=365)

    forecaster = DemandForecaster().fit(df)
    print("=== Demand Forecaster Evaluation ===")
    forecaster.evaluate()

    # --- Elasticity curve for one product ---
    product_id = 3
    current_price = df[df.product_id == product_id].price.mean()
    elasticity_est = PriceElasticityEstimator(forecaster)
    price_grid = np.linspace(current_price * 0.6, current_price * 1.4, 15)
    curve = elasticity_est.estimate_curve(product_id, price_grid, is_weekend=0, season_factor=1.0)
    print(f"\n=== Estimated Demand Curve for Product {product_id} (current avg price ${current_price:.2f}) ===")
    print(curve.to_string(index=False))

    # --- Price optimization ---
    optimizer = PriceOptimizer(forecaster, unit_cost_ratio=0.55)
    result = optimizer.optimize(
        product_id, current_price, objective="profit",
        is_weekend=0, season_factor=1.0, promo=0
    )
    print(f"\n=== Price Optimization Result (objective: {result['objective']}) ===")
    for k, v in result.items():
        if k != "full_grid":
            print(f"{k}: {v}")

    print("\nTop 5 candidate prices by projected profit:")
    print(result["full_grid"].sort_values("profit", ascending=False).head(5).to_string(index=False))

=== Demand Forecaster Evaluation ===
Demand Forecast — MAE: 17.48 units | MAPE: 14.5%

=== Estimated Demand Curve for Product 3 (current avg price $87.91) ===
     price  predicted_demand
 52.743521        184.760762
 57.766714        183.047488
 62.789906        182.059791
 67.813099        187.645562
 72.836291        196.997448
 77.859484        193.386784
 82.882676        175.663542
 87.905869        166.836752
 92.929061        151.311633
 97.952254        133.437036
102.975446        133.365482
107.998639        131.416332
113.021831        128.722982
118.045024        128.261775
123.068216        108.950683

=== Price Optimization Result (objective: profit) ===
recommended_price: 109.88
current_price: 87.90586883142868
price_change_pct: 25.0
predicted_demand_at_recommended_price: 130.9
objective: profit
objective_value: 8052.43

Top 5 candidate prices by projected profit:
     price  predicted_demand      revenue      profit
109.882336        130.861274 14379.342525 8052.431814